# 10 — Next Time Prediction

Regression: given a prefix of k events, predict the number of **days until the start of the next event** (`iloc[k]`).  
Target = `(timestamp of event k) − (timestamp of event k-1)` in days (always ≥ 0).  
Methodology follows Teinemaa et al. (2019): prefix-length bucketing [1, 3, 5, 10, 20] × Boolean/Succession encodings × ML sweep, then LSTM + Transformer regressors.  
Metrics: MAE, RMSE, R², MedAE.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
import pm4py as pm
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

## 2. Load & Clean

In [ ]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv('data/'+
    'event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}')
print(f'Cases: {df["Case_id"].nunique():,}')

## 3. Format for pm4py + Chronological 60/20/20 Split

In [ ]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)

case_start = df.groupby('case:concept:name')['time:timestamp'].min().sort_values()
n_cases      = len(case_start)
train_cutoff = case_start.iloc[int(n_cases * 0.60)]
val_cutoff   = case_start.iloc[int(n_cases * 0.80)]

train_cases = case_start[case_start <  train_cutoff].index
val_cases   = case_start[(case_start >= train_cutoff) & (case_start < val_cutoff)].index
test_cases  = case_start[case_start >= val_cutoff].index

print(f'Train: {len(train_cases):,} | Val: {len(val_cases):,} | Test: {len(test_cases):,}')
print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')

train_df = df[df['case:concept:name'].isin(train_cases)].copy()
val_df   = df[df['case:concept:name'].isin(val_cases)].copy()
test_df  = df[df['case:concept:name'].isin(test_cases)].copy()

## 4. Remove Leaky Activities + Build Vocabulary

In [ ]:
LEAKY_THRESHOLD = 0.85
act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)
print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

all_train_acts = sorted(train_df['concept:name'].dropna().unique())
act2idx = {act: i+2 for i, act in enumerate(all_train_acts)}
act2idx['<PAD>'] = 0
act2idx['<UNK>'] = 1
idx2act = {v: k for k, v in act2idx.items()}
VOCAB_SIZE  = len(act2idx)
MAX_SEQ_LEN = 30
print(f'\nVocabulary size: {VOCAB_SIZE} | MAX_SEQ_LEN: {MAX_SEQ_LEN}')

## 5. Next-Time Label Extraction

Target for prefix k: time delta between event k-1 and event k (the **next** event after the prefix end).  
Cases with ≤ k events (no next event) are excluded.

In [ ]:
def extract_next_time_labels(df_subset, k):
    """
    Return Series: case_id -> days between prefix last event (k-1) and next event (k).
    Clipped to [0, inf) to handle near-simultaneous events.
    """
    records = {}
    for cid, grp in df_subset.groupby('case:concept:name', sort=False):
        grp = grp.sort_values('time:timestamp')
        if len(grp) <= k:
            continue  # no next event
        t_prev = grp.iloc[k-1]['time:timestamp']  # last prefix event
        t_next = grp.iloc[k]['time:timestamp']     # next event (target)
        delta  = (t_next - t_prev).total_seconds() / 86400.0
        records[cid] = max(0.0, delta)
    return pd.Series(records, name='next_time_days')


def describe_target(y_series, label):
    print(f'{label}: n={len(y_series):,}  mean={y_series.mean():.2f}d  '
          f'median={y_series.median():.2f}d  p95={y_series.quantile(0.95):.2f}d  '
          f'max={y_series.max():.1f}d')


# Quick sanity check on a single prefix length
sample_labels = extract_next_time_labels(train_df, k=5)
describe_target(sample_labels, 'Next time (k=5, train)')

## 6. Encoding Helpers (Boolean / Succession)

In [ ]:
PM_COLS        = ['case:concept:name', 'concept:name', 'time:timestamp']
SKIP_COLS      = {'case:concept:name', 'concept:name', 'time:timestamp', '@@diff_start_end'}
CASE_ATTR_COLS = ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']

case_attrs_train = train_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_val   = val_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_test  = test_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)


def get_prefix_pm(df_clean, k):
    prefix = pm.get_prefixes_from_log(df_clean, length=k, case_id_key='case:concept:name')
    return prefix[PM_COLS].copy()


def bool_encoding(prefix_pm):
    enriched = pm.extract_outcome_enriched_dataframe(
        prefix_pm, activity_key='concept:name',
        timestamp_key='time:timestamp', case_id_key='case:concept:name',
    )
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return enriched.drop_duplicates('case:concept:name').set_index('case:concept:name')[feat_cols].sort_index()


def succ_encoding(prefix_pm):
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())
    df_s = prefix_pm.copy()
    df_s['prev_act'] = df_s.groupby('case:concept:name')['concept:name'].shift(1)
    df_s = df_s.dropna(subset=['prev_act'])
    if df_s.empty:
        return pd.DataFrame(index=pd.Index(all_case_ids, name='case:concept:name'))
    df_s['bigram'] = df_s['prev_act'] + ' >> ' + df_s['concept:name']
    bigram_counts = df_s.groupby(['case:concept:name', 'bigram']).size().unstack(fill_value=0)
    return bigram_counts.reindex(all_case_ids, fill_value=0).rename_axis('case:concept:name')


def build_Xy(X_pm, case_attrs_split, y_series, train_cols=None):
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    common = X.index.intersection(y_series.index)
    X = X.loc[common]
    y = y_series.reindex(common).values.astype(np.float32)
    return X.values.astype(np.float32), y, list(X.columns)


def regression_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    med  = median_absolute_error(y_true, y_pred)
    if label:
        print(f'  {label:<20s}  MAE={mae:.3f}d  RMSE={rmse:.3f}d  R²={r2:.4f}  MedAE={med:.3f}d')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MedAE': med}


print('Encoding helpers defined.')

## 7. Prefix-Length Sweep — XGBoost + LightGBM

In [ ]:
PREFIX_LENGTHS = [1, 3, 5, 10, 20]
ENCODINGS      = [('Boolean', bool_encoding), ('Succession', succ_encoding)]
sweep_results  = {}  # (enc_name, model_name, k) -> metrics

for enc_name, enc_fn in ENCODINGS:
    print(f'\n── {enc_name} encoding ──────────────────────')
    for k in PREFIX_LENGTHS:
        train_prefix_pm = get_prefix_pm(train_df, k)
        val_prefix_pm   = get_prefix_pm(val_df,   k)
        test_prefix_pm  = get_prefix_pm(test_df,  k)

        X_tr_pm    = enc_fn(train_prefix_pm)
        train_cols = list(X_tr_pm.columns)
        X_va_pm    = enc_fn(val_prefix_pm)
        X_te_pm    = enc_fn(test_prefix_pm)

        y_tr_s = extract_next_time_labels(train_df, k)
        y_va_s = extract_next_time_labels(val_df,   k)
        y_te_s = extract_next_time_labels(test_df,  k)

        X_tr, y_tr, _ = build_Xy(X_tr_pm, case_attrs_train, y_tr_s)
        X_va, y_va, _ = build_Xy(X_va_pm, case_attrs_val,   y_va_s, train_cols=train_cols)
        X_te, y_te, _ = build_Xy(X_te_pm, case_attrs_test,  y_te_s, train_cols=train_cols)

        if len(X_tr) == 0 or len(X_te) == 0:
            print(f'  k={k}: skipping (empty split)')
            continue

        describe_target(pd.Series(y_tr), f'  {enc_name} k={k} train')

        for model_name, reg in [
            ('XGB',  XGBRegressor(n_estimators=500, early_stopping_rounds=20,
                                  n_jobs=-1, random_state=RANDOM_STATE, verbosity=0)),
            ('LGBM', LGBMRegressor(n_estimators=500, n_jobs=-1,
                                   random_state=RANDOM_STATE, verbose=-1)),
        ]:
            if model_name == 'XGB':
                reg.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                reg.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                        callbacks=[lgb.early_stopping(20, verbose=False),
                                   lgb.log_evaluation(period=-1)])
            pred = reg.predict(X_te)
            m    = regression_metrics(y_te, pred, label=f'{enc_name} k={k} {model_name}')
            sweep_results[(enc_name, model_name, k)] = m

## 8. Deep Learning — LSTM + Transformer Regressors

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(torch.__version__)

k_dl = 10  # representative prefix length for DL models


def build_sequences(df_subset, case_ids, k, act2idx, max_seq_len):
    seqs = []
    df_s = df_subset.sort_values(['case:concept:name', 'time:timestamp'])
    grp_map = {cid: grp for cid, grp in df_s.groupby('case:concept:name', sort=False)}
    for cid in case_ids:
        grp = grp_map.get(cid)
        if grp is None:
            seqs.append([0] * max_seq_len)
            continue
        acts = grp['concept:name'].tolist()[:k]
        idxs = [act2idx.get(a, act2idx['<UNK>']) for a in acts]
        seq  = idxs[-max_seq_len:]
        seqs.append([0] * (max_seq_len - len(seq)) + seq)
    return np.array(seqs, dtype=np.int64)


# Labels and features at k_dl
y_tr_dl = extract_next_time_labels(train_df, k_dl)
y_va_dl = extract_next_time_labels(val_df,   k_dl)
y_te_dl = extract_next_time_labels(test_df,  k_dl)

X_bool_tr_pm = bool_encoding(get_prefix_pm(train_df, k_dl))
bool_cols    = list(X_bool_tr_pm.columns)
X_bool_tr, y_tr_tab, _ = build_Xy(X_bool_tr_pm, case_attrs_train, y_tr_dl)
X_bool_va, y_va_tab, _ = build_Xy(bool_encoding(get_prefix_pm(val_df,  k_dl)),
                                   case_attrs_val,  y_va_dl, bool_cols)
X_bool_te, y_te_tab, _ = build_Xy(bool_encoding(get_prefix_pm(test_df, k_dl)),
                                   case_attrs_test, y_te_dl, bool_cols)

scaler = MinMaxScaler()
X_tr_tab = scaler.fit_transform(X_bool_tr).astype(np.float32)
X_va_tab = scaler.transform(X_bool_va).astype(np.float32)
X_te_tab = scaler.transform(X_bool_te).astype(np.float32)
N_TAB = X_tr_tab.shape[1]

seqs_tr = build_sequences(train_df, y_tr_dl.index, k_dl, act2idx, MAX_SEQ_LEN)
seqs_va = build_sequences(val_df,   y_va_dl.index, k_dl, act2idx, MAX_SEQ_LEN)
seqs_te = build_sequences(test_df,  y_te_dl.index, k_dl, act2idx, MAX_SEQ_LEN)


class NextTimeDataset(Dataset):
    def __init__(self, X, seqs, y):
        self.X    = torch.tensor(X,    dtype=torch.float32)
        self.seqs = torch.tensor(seqs, dtype=torch.long)
        self.y    = torch.tensor(y,    dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.seqs[i], self.y[i]


train_ds = NextTimeDataset(X_tr_tab, seqs_tr, y_tr_tab)
val_ds   = NextTimeDataset(X_va_tab, seqs_va, y_va_tab)
test_ds  = NextTimeDataset(X_te_tab, seqs_te, y_te_tab)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
class LSTMNextTime(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=128, n_layers=2, dropout=0.3, n_tab=10):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc1  = nn.Linear(hidden + n_tab, 64)
        self.fc2  = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        _, (h, _) = self.lstm(self.emb(seq))
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([h[-1], tab], dim=1)))))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerNextTime(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, nhead=4, n_layers=2,
                 d_ff=128, dropout=0.1, n_tab=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.pe  = PositionalEncoding(emb_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                               dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.fc1 = nn.Linear(emb_dim + n_tab, 64)
        self.fc2 = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        pad_mask = (seq == 0)
        x = self.pe(self.emb(seq))
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).unsqueeze(-1).float()
        x = (x * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([x, tab], dim=1)))))


def train_reg_model(model, train_loader, val_loader, n_epochs=20):
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_mae, best_state, no_improve = float('inf'), None, 0

    for epoch in range(n_epochs):
        model.train()
        for X_b, seq_b, y_b in train_loader:
            X_b, seq_b, y_b = X_b.to(device), seq_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(model(seq_b, X_b), y_b).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds_v, targets_v = [], []
        with torch.no_grad():
            for X_b, seq_b, y_b in val_loader:
                X_b, seq_b = X_b.to(device), seq_b.to(device)
                preds_v.extend(model(seq_b, X_b).cpu().numpy().ravel())
                targets_v.extend(y_b.numpy().ravel())
        val_mae = mean_absolute_error(targets_v, preds_v)
        scheduler.step(val_mae)
        print(f'  Epoch {epoch+1:02d}  val_MAE={val_mae:.4f}d')

        if val_mae < best_mae:
            best_mae   = val_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f'  Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


print('DL model definitions ready.')

In [ ]:
dl_results = {}

lstm_model = LSTMNextTime(VOCAB_SIZE, n_tab=N_TAB).to(device)
print('Training LSTM...')
lstm_model = train_reg_model(lstm_model, train_loader, val_loader)
lstm_model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        all_preds.extend(lstm_model(seq_b, X_b).cpu().numpy().ravel())
        all_targets.extend(y_b.numpy().ravel())
dl_results['LSTM'] = regression_metrics(all_targets, all_preds, label='LSTM')

tf_model = TransformerNextTime(VOCAB_SIZE, n_tab=N_TAB).to(device)
print('\nTraining Transformer...')
tf_model = train_reg_model(tf_model, train_loader, val_loader)
tf_model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        all_preds.extend(tf_model(seq_b, X_b).cpu().numpy().ravel())
        all_targets.extend(y_b.numpy().ravel())
dl_results['Transformer'] = regression_metrics(all_targets, all_preds, label='Transformer')

## 9. Results Table + MAE vs Prefix Plot

In [ ]:
print('\n=== SWEEP RESULTS ===')
for (enc, mname, k), m in sorted(sweep_results.items(), key=lambda x: (x[0][0], x[0][1], x[0][2])):
    print(f'{enc:12s}  {mname:6s}  k={str(k):3s}  '
          f'MAE={m["MAE"]:.3f}d  RMSE={m["RMSE"]:.3f}d  R²={m["R2"]:.4f}  MedAE={m["MedAE"]:.3f}d')

print(f'\n=== DL MODELS (k={k_dl}) ===')
for mname, m in dl_results.items():
    print(f'{mname:14s}  MAE={m["MAE"]:.3f}d  RMSE={m["RMSE"]:.3f}d  R²={m["R2"]:.4f}  MedAE={m["MedAE"]:.3f}d')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for enc_name, _ in ENCODINGS:
    for model_name in ['XGB', 'LGBM']:
        maes  = [sweep_results.get((enc_name, model_name, k), {}).get('MAE',  np.nan) for k in PREFIX_LENGTHS]
        rmses = [sweep_results.get((enc_name, model_name, k), {}).get('RMSE', np.nan) for k in PREFIX_LENGTHS]
        label = f'{enc_name}/{model_name}'
        axes[0].plot([str(k) for k in PREFIX_LENGTHS], maes,  marker='o', label=label)
        axes[1].plot([str(k) for k in PREFIX_LENGTHS], rmses, marker='o', label=label)

axes[0].set_title('Next Time: MAE vs Prefix Length')
axes[0].set_xlabel('Prefix Length'); axes[0].set_ylabel('MAE (days)'); axes[0].legend(fontsize=8)
axes[1].set_title('Next Time: RMSE vs Prefix Length')
axes[1].set_xlabel('Prefix Length'); axes[1].set_ylabel('RMSE (days)'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('figs/next_time_metrics_vs_prefix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/next_time_metrics_vs_prefix.png')

In [ ]:
# Predicted vs Actual scatter (Transformer, clipped to 90th percentile for readability)
clip_val = np.percentile(all_targets, 90)
y_c = np.clip(all_targets, 0, clip_val)
p_c = np.clip(all_preds,   0, clip_val)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_c, p_c, alpha=0.1, s=5)
ax.plot([0, clip_val], [0, clip_val], 'r--', lw=1)
ax.set_xlabel('Actual (days)'); ax.set_ylabel('Predicted (days)')
ax.set_title(f'Next Time — Transformer pred vs actual (k={k_dl}, clipped at p90={clip_val:.1f}d)')
plt.tight_layout()
plt.savefig('figs/next_time_pred_vs_actual.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/next_time_pred_vs_actual.png')